<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "266"
iso_country = "Gabon"
year = 2023
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 20.060*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Gabon': 

Number of trade records present in dataset: 31697
Number of partner countries present in dataset: 176
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$2830.4m
Total weight within the dataset: 1614.8 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,266,Gabon,056,Belgium,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",222.622023,NaN,222.622023,kg,257.854,257.854,37.088829,8413.0,5227.594223,193.885348,['Container'],['Container'],0.693981,0.691623,0.691803,0.691803,0.693260,0.693260,0.728695,1.359349,0.803117,0.742630,0.745366,0.76613,0.712215,1.298004,0.839246,0.772106,0.754298,0.791786,0.697774,1.435916,1.052538,0.916777,0.847205,0.827162,0.683544,1.523815,1.227098,1.145156,0.898897,0.889193,0.669576,1.553328,1.469693,1.493560,0.972715,0.935134,0.655337,1.525757,1.45800,1.555055,1.167885,0.952827,0.641967,1.351797,1.397462,1.459839,1.082056,0.96442,0.628336,1.222299,1.248389,1.455613,1.205115,0.971830,3693
1,1,266,Gabon,380,Italy,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",911.107967,NaN,911.107967,kg,278.122,278.122,151.790587,7875.0,4893.296625,742.756369,['Container'],['Container'],2.658574,2.649544,2.650234,2.650234,2.655815,2.655815,2.791563,5.207538,3.076663,2.844946,2.855428,2.93497,2.728429,4.972528,3.215072,2.957864,2.889642,3.033256,2.673106,5.500859,4.032172,3.512086,3.245563,3.168778,2.618592,5.837591,4.700896,4.386985,3.443590,3.406413,2.565081,5.950653,5.630253,5.721688,3.726380,3.582410,2.510535,5.845031,5.58546,5.957267,4.474058,3.650188,2.459315,5.178606,5.353543,5.592504,4.145252,3.69460,2.407097,4.682510,4.782461,5.576315,4.616681,3.722989,3693


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $450.5m (15.9%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
02 | Meat and edible meat  126588147.9   61551045.3      4.5      3.8
10 | Cereals                87225835.9  213945927.4      3.1     13.2
19 | Preparations of cere   39262006.3   13581802.0      1.4      0.8
16 | Meat, fish, crustace   34203575.5    7424885.1      1.2      0.5
04 | Dairy produce; birds   33544473.0   14094816.9      1.2      0.9
11 | Products of the mill   27729171.5   54299039.3      1.0      3.4
22 | Beverages, spirits a   24176176.0   37749248.4      0.9      2.3
03 | Fish and crustaceans   23746070.5   10892748.2      0.8      0.7
20 | Preparations of vege   15475574.7   12492385.2      0.5      0.8
21 | Miscellaneous edible   12459673.4    4038354.4      0.4      0.3
17 | Sugars and sugar con    7374574.0   14639715.3      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $390.1m (13.8%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       252243920.6  233414450.3      8.9     14.5
Food & Beverages  136754815.8   90886057.8      4.8      5.6
Fishing             1089504.1     228955.6      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $346.9m (12.3%), ... 

                                  cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
OMT - Meat products n.e.c.     115422202.0  56482216.3      4.1      3.5
OFD - Food products n.e.c.     105628218.8  55474127.9      3.7      3.4
PCR - Processed rice            54186989.9  94122801.5      1.9      5.8
MIL - Dairy products            32518752.0  12747651.8      1.1      0.8
CMT - Bovine meat prods         20955374.0   7226151.2      0.7      0.4
VOL - Vegetable oils and fats    7776445.2   9035048.0      0.3      0.6
SGR - Sugar                      6353132.7  13932700.1      0.2      0.9
V_F - Vegetables, fruit, nuts    2755352.4   3418793.3      0.1      0.2
OCR - Crops n.e.c.                538275.3    166387.4      0.0      0.0
OAP - Animal products n.e.c.      469279.2    114546.7      0.0      0.0
GRO - Cereal grains n.e.c.        108417.6    428163.4    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 15.9% 
GTAP 12.3% 
EORA 13.8%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46369/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,266,Gabon,056,Belgium,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",222.622023,NaN,222.622023,kg,257.854,257.854,37.088829,8413.0,5227.594223,193.885348,['Container'],['Container'],0.693981,0.691623,0.691803,0.691803,0.693260,0.693260,0.728695,1.359349,0.803117,0.742630,0.745366,0.76613,0.712215,1.298004,0.839246,0.772106,0.754298,0.791786,0.697774,1.435916,1.052538,0.916777,0.847205,0.827162,0.683544,1.523815,1.227098,1.145156,0.898897,0.889193,0.669576,1.553328,1.469693,1.493560,0.972715,0.935134,0.655337,1.525757,1.45800,1.555055,1.167885,0.952827,0.641967,1.351797,1.397462,1.459839,1.082056,0.96442,0.628336,1.222299,1.248389,1.455613,1.205115,0.971830,3693,NaN
1,1,266,Gabon,380,Italy,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",911.107967,NaN,911.107967,kg,278.122,278.122,151.790587,7875.0,4893.296625,742.756369,['Container'],['Container'],2.658574,2.649544,2.650234,2.650234,2.655815,2.655815,2.791563,5.207538,3.076663,2.844946,2.855428,2.93497,2.728429,4.972528,3.215072,2.957864,2.889642,3.033256,2.673106,5.500859,4.032172,3.512086,3.245563,3.168778,2.618592,5.837591,4.700896,4.386985,3.443590,3.406413,2.565081,5.950653,5.630253,5.721688,3.726380,3.582410,2.510535,5.845031,5.58546,5.957267,4.474058,3.650188,2.459315,5.178606,5.353543,5.592504,4.145252,3.69460,2.407097,4.682510,4.782461,5.576315,4.616681,3.722989,3693,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $318.2m of $2830.4m (11.2%)
Volume: 363.2 of 1614.8 million kilograms (22.5%)
Value-to-Weight: 0.88 from 1.75 US$ per kg (+-50.0%)
Average distance to market (volume-weighted): 6359.9 from 6668.9 nm (+-4.6%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    820  128165093.5  150694854.0      4.5      9.3
1        Meat and Prep    274  105489953.0   54289513.8      3.7      3.4
2                Dairy    232   33379874.7   14034113.7      1.2      0.9
3  Unprocessed Cereals     31   33030706.7  119808987.9      1.2      7.4
4      Sugar and Honey     98    7403486.9   14648004.6      0.3      0.9
5        Fats and Oils    137    4342839.9    3192704.6      0.2      0.2
6           Vegetables    500    4268666.9    4434694.1      0.2      0.3
7               Fruits    385    2120262.0    2063782.9      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,31104496.4,57806270.9,34244140.5,31684820.0,31800738.1,32689918.7,28599488.1,6.614415e+07,62556468.1,63608584.0,41472815.6,39855605.0,26851136.2,52058342.1,53169360.6,61954538.3,51323828.9,41411989.5
1,Fats and Oils,6697102.6,12442462.5,7372601.9,6821681.6,6846691.4,7038378.3,6157899.2,1.424167e+07,13467690.4,13694140.2,8928785.4,8581121.9,5781547.5,11205142.5,11445395.5,13335625.6,11048897.1,8915926.4
2,Fruits,3568382.7,6462188.5,3899755.6,3623690.7,3636194.9,3740416.6,3295660.0,7.451628e+06,7030982.4,7178704.6,4721173.7,4523108.9,3104713.9,5883598.0,6007912.6,6971299.5,5794921.8,4693641.6
3,Meat and Prep,108469170.1,202344490.3,119547062.3,110543447.5,110950723.7,114041393.2,99668977.2,2.312190e+08,218769523.8,222322325.3,144792495.9,139198377.2,93530352.3,181943974.0,185827673.3,216673706.8,179386093.1,144660752.1
4,Processed Cereals,471067886.2,876912780.8,518862253.4,479941854.2,481704551.6,495159581.1,433009166.4,1.002632e+09,948635940.0,964289209.9,628609557.9,603958825.4,406457749.1,789272139.0,806074462.7,939574255.6,778114676.5,627573710.6
5,Sugar and Honey,16489814.1,29130736.9,17893391.9,16677940.7,16736247.1,17241024.3,15293161.8,3.376022e+07,32060910.8,32747564.9,21972998.3,20642321.3,14458008.1,26934478.9,27444653.3,31747325.3,26494906.9,21362425.0
6,Unprocessed Cereals,217879486.5,406325584.9,240110892.1,222036687.8,222854523.2,229065273.1,200213056.9,4.643424e+08,439350443.1,446497201.5,290839507.3,279566462.4,187889725.2,365414076.8,373209142.3,435140758.7,360272366.5,290530441.3
7,Vegetables,8848586.9,15782887.0,9627388.2,8968371.6,8998627.5,9260821.4,8194411.6,1.828110e+07,17241918.3,17637215.2,11662489.6,11148823.7,7731350.3,14458468.1,14768221.2,17109097.2,14226835.7,11558235.2


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.80,0.09,0.02,0.02,0.05,1.12,1.02,1.05,0.39,0.34,0.76,0.79,1.05,0.73,0.44
1,Fats and Oils,1.32,0.16,0.03,0.03,0.08,1.86,1.68,1.74,0.64,0.56,1.25,1.30,1.74,1.21,0.72
2,Fruits,1.36,0.16,0.03,0.03,0.08,1.96,1.76,1.83,0.67,0.58,1.31,1.37,1.82,1.27,0.75
3,Meat and Prep,0.89,0.11,0.02,0.02,0.05,1.25,1.13,1.16,0.43,0.37,0.84,0.87,1.17,0.81,0.48
4,Processed Cereals,3.17,0.37,0.07,0.08,0.19,4.44,4.02,4.15,1.53,1.33,2.99,3.12,4.16,2.90,1.73
5,Sugar and Honey,1.71,0.19,0.03,0.03,0.10,2.49,2.26,2.36,0.90,0.72,1.69,1.75,2.34,1.63,0.93
6,Unprocessed Cereals,5.71,0.67,0.13,0.15,0.34,8.00,7.24,7.46,2.74,2.40,5.37,5.61,7.49,5.22,3.11
7,Vegetables,1.62,0.18,0.03,0.04,0.10,2.36,2.12,2.21,0.81,0.69,1.58,1.65,2.20,1.52,0.90
